In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/anushkumar117/legalllll/val.csv
/kaggle/input/datasets/anushkumar117/legalllll/train.csv
/kaggle/input/datasets/anushkumar117/legalllll/test.csv


In [2]:
!pip install transformers datasets pandas scikit-learn accelerate

import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ======================================================
# CHANGE ONLY THESE FOR ANY DATASET
# ======================================================
TEXT_COLUMN = "email"      # sentiment = "text" | spam = "email"
LABEL_COLUMN = "label"

# OUTPUT LABELS
LABEL_0_NAME = "Not Spam"
LABEL_1_NAME = "Spam"

# For Sentiment use:
# LABEL_0_NAME = "Negative"
# LABEL_1_NAME = "Positive"

# ======================================================
# LOAD DATA
# ======================================================
train_df = pd.read_csv("/kaggle/input/datasets/anushkumar117/legalllll/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/anushkumar117/legalllll/test.csv")
valid_df = pd.read_csv("/kaggle/input/datasets/anushkumar117/legalllll/val.csv")

# ======================================================
# BASIC CHECKS
# ======================================================
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)
valid_df.drop_duplicates(inplace=True)

# ======================================================
# CLEAN TEXT COLUMN
# ======================================================
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].fillna("").astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].fillna("").astype(str)
valid_df[TEXT_COLUMN] = valid_df[TEXT_COLUMN].fillna("").astype(str)

# ======================================================
# CONVERT TO DATASET
# ======================================================
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
valid_dataset = Dataset.from_pandas(valid_df)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset
})

# ======================================================
# TOKENIZER
# ======================================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    texts = [str(x) for x in example[TEXT_COLUMN]]

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COLUMN])
tokenized_datasets = tokenized_datasets.rename_column(LABEL_COLUMN, "labels")
tokenized_datasets.set_format("torch")

# ======================================================
# MODEL
# ======================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="no",
    logging_dir="./logs",
    logging_steps=200,
    dataloader_pin_memory=False
)

# ======================================================
# METRICS
# ======================================================
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# ======================================================
# TRAINER
# ======================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

trainer.evaluate()

# ======================================================
# SINGLE TEXT PREDICTION
# ======================================================
def predict(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return LABEL_1_NAME if predicted_class_id == 1 else LABEL_0_NAME

# ======================================================
# CUSTOM INPUT
# ======================================================
user_review = input("Enter Text: ")
print(predict(user_review))

# ======================================================
# TEST PREDICTIONS + SUBMISSION
# ======================================================
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame({
    TEXT_COLUMN: test_df[TEXT_COLUMN],
    "predicted_label": predicted_labels
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print(submission.head())
'''
# ======================================================
# TEXT LABEL SUBMISSION
# ======================================================
label_map = {
    0: LABEL_0_NAME,
    1: LABEL_1_NAME
}

predicted_names = [label_map[x] for x in predicted_labels]

submission = pd.DataFrame({
    TEXT_COLUMN: test_df[TEXT_COLUMN],
    "predicted_label": predicted_names
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print(submission.head())
'''

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2325 [00:00<?, ? examples/s]

...

Enter Text:  hello


Not Spam


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


submission.csv created successfully!
                                               email  predicted_label
0   if the text recognition algorithm architectur...                0
1  hi i m trying to build sa under digital unix N...                0
2  owne byrne sure if you re willing to risk firi...                0
3  i think that this and other articles confuse s...                0
4   it seems that something changed during the la...                0


'\n# ======================================================\n# TEXT LABEL SUBMISSION\n# ======================================================\nlabel_map = {\n    0: LABEL_0_NAME,\n    1: LABEL_1_NAME\n}\n\npredicted_names = [label_map[x] for x in predicted_labels]\n\nsubmission = pd.DataFrame({\n    TEXT_COLUMN: test_df[TEXT_COLUMN],\n    "predicted_label": predicted_names\n})\n\nsubmission.to_csv("submission.csv", index=False)\n\nprint("submission.csv created successfully!")\nprint(submission.head())\n'

In [4]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.1528269648551941, 'eval_accuracy': 0.9764309764309764, 'eval_f1': 0.9195402298850575, 'eval_precision': 0.975609756097561, 'eval_recall': 0.8695652173913043, 'eval_runtime': 1.4382, 'eval_samples_per_second': 206.502, 'eval_steps_per_second': 6.953, 'epoch': 2.0}
